# Titanic Passengers

Scraped from Wikipedia

* https://www.kaggle.com/code/pavlofesenko/extending-titanic-dataset-using-wikipedia
* https://pandas.pydata.org/docs/reference/api/pandas.read_html.html
* https://pandas.pydata.org/docs/user_guide/io.html#io-read-html
* https://en.wikipedia.org/wiki/Passengers_of_the_Titanic
* https://en.wikipedia.org/wiki/Crew_of_the_Titanic

In [1]:
# pip install html5lib

In [2]:
import pandas as pd

first_class = pd.read_html('https://en.wikipedia.org/wiki/Passengers_of_the_Titanic', header=0)[1]
first_class['pclass'] = 1
first_class.columns = first_class.columns.str.strip().str.lower()
print(first_class.shape)
first_class.head()

(326, 8)


,name,age,hometown,boarded,destination,lifeboat,body,pclass
0,"Allen, Miss Elizabeth Walton",29,"St Louis, Missouri, US",Southampton,St Louis,NaN,NaN,1
1,"Allison, Mr. Hudson Creighton",30,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,135MB,1
2,"and chauffeur, Mr. George Swane[70]",19,NaN,Southampton,"Montreal, Quebec, Canada",NaN,294MB,1
3,"and cook, Miss Amelia Mary ""Mildred"" Brown[70]",18,"London, England, UK",Southampton,"Montreal, Quebec, Canada",11,NaN,1
4,"Allison, Mrs. Bessie Waldo (née Daniels)",25,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,NaN,1


In [3]:
from bs4 import BeautifulSoup
import pandas as pd
import requests

soup = BeautifulSoup(requests.get("https://en.wikipedia.org/wiki/Passengers_of_the_Titanic").text)
s = soup.select_one('table:nth-of-type(2)').find_all('tr', {'style': 'background:#9bddff;'})
s_list = [x.text.split('\n')[1] for x in s]
s_series = pd.Series(s_list)
first_class_survivors = s_series.to_frame()
first_class_survivors.columns = ['survivors']
first_class_survivors.head()

,survivors
0,"Allen, Miss Elizabeth Walton"
1,"and cook, Miss Amelia Mary ""Mildred"" Brown[70]"
2,"and maid, Miss Sarah Daniels"
3,"Allison, Master Hudson Trevor"
4,"and nurse, Miss Alice Catherine Cleaver"


In [4]:
second_class = pd.read_html('https://en.wikipedia.org/wiki/Passengers_of_the_Titanic', header=0)[2]
second_class['pclass'] = 2
second_class.columns = second_class.columns.str.strip().str.lower()
print(second_class.shape)
second_class.head()

(278, 8)


,name,age,hometown,boarded,destination,lifeboat,body,pclass
0,"Abelson, Mr. Samuel",30,Russia,Cherbourg,"New York, New York, US",NaN,NaN,2
1,"Abelson, Mrs. Anna (née Wizosky?)",28,Russia,Cherbourg,"New York, New York, US",10,NaN,2
2,"Andrew, Mr. Edgar Samuel",17,"San Ambrosio, Córdoba, Argentina",Southampton,"Trenton, New Jersey, US",NaN,NaN,2
3,"Andrew, Mr. Frank Thomas",30,"Redruth, Cornwall, England",Southampton,"Houghton, Michigan, US",NaN,NaN,2
4,"Angle, Mr. William A.",32,"Warwick, Warwickshire, England",Southampton,New York City,NaN,NaN,2


In [5]:
third_class = pd.read_html('https://en.wikipedia.org/wiki/Passengers_of_the_Titanic', header=0)[3]
third_class['pclass'] = 3
third_class.columns = third_class.columns.str.strip().str.lower()
third_class.hometown = third_class['hometown'] + ', ' + third_class['home country']
third_class.drop('home country', axis=1, inplace=True)
print(third_class.shape)
third_class.head()

(708, 8)


,name,age,hometown,boarded,destination,lifeboat,body,pclass
0,"Abbing, Mr. Anthony",40,"Cincinnati, Ohio, US",Southampton,"Cincinnati, Ohio, US",NaN,NaN,3
1,"Abbott, Mrs. Rhoda Mary (née Hunt)",39,"East Providence, Rhode Island, US",Southampton,"East Providence, Rhode Island, US",A,NaN,3
2,"Abbott, Mr. Rossmore Edward",16,"East Providence, Rhode Island, US",Southampton,"East Providence, Rhode Island, US",NaN,190MB,3
3,"Abbott, Mr. Eugene Joseph",14,"East Providence, Rhode Island, US",Southampton,"East Providence, Rhode Island, US",NaN,NaN,3
4,"Abd al-Khaliq, Mr. Farid Qasim Husayn",18,"Shana, Syria",Cherbourg,New York City,NaN,NaN,3


## Concat

In [6]:
passengers = pd.concat([first_class, second_class, third_class], ignore_index=True)
print(passengers.shape)
print(passengers.info())
passengers.sample(5)

(1312, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1312 entries, 0 to 1311
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         1296 non-null   object
 1   age          1312 non-null   object
 2   hometown     1272 non-null   object
 3   boarded      1309 non-null   object
 4   destination  1311 non-null   object
 5   lifeboat     502 non-null    object
 6   body         133 non-null    object
 7   pclass       1312 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 82.1+ KB
None


,name,age,hometown,boarded,destination,lifeboat,body,pclass
859,"Hart, Mr. Henry John",27,"Ballysadare, Sligo, Ireland",Queenstown,"Boston, Massachusetts, US",NaN,NaN,3
839,"Guest, Mr. Robert",32,"London, England",Southampton,"Clinton, New York, US",NaN,NaN,3
1030,"Mockler, Miss Ellen Mary",23,"Currafarry, Galway, Ireland",Queenstown,New York City,16,NaN,3
991,"Madsen, Mr. Fridtjof Arne",24,"Trondheim, Norway",Southampton,"Brooklyn, New York, US",13,NaN,3
1060,"Nasr Alma, Mr. Mustafa",20,"Tebnine, Syria",Cherbourg,New York City,NaN,NaN,3


In [7]:
passengers.isnull().sum()

name             16
age               0
hometown         40
boarded           3
destination       1
lifeboat        810
body           1179
pclass            0
dtype: int64

In [8]:
# extracting surnames
surnames = passengers.name.str.extract(r'(?P<surname>.*), (?P<title>[^\s]+) (?P<fmname>.*)')
passengers = pd.concat([passengers, surnames], axis=1)
passengers.head()

,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname
0,"Allen, Miss Elizabeth Walton",29,"St Louis, Missouri, US",Southampton,St Louis,NaN,NaN,1,Allen,Miss,Elizabeth Walton
1,"Allison, Mr. Hudson Creighton",30,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,135MB,1,Allison,Mr.,Hudson Creighton
2,"and chauffeur, Mr. George Swane[70]",19,NaN,Southampton,"Montreal, Quebec, Canada",NaN,294MB,1,and chauffeur,Mr.,George Swane[70]
3,"and cook, Miss Amelia Mary ""Mildred"" Brown[70]",18,"London, England, UK",Southampton,"Montreal, Quebec, Canada",11,NaN,1,and cook,Miss,"Amelia Mary ""Mildred"" Brown[70]"
4,"Allison, Mrs. Bessie Waldo (née Daniels)",25,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,NaN,1,Allison,Mrs.,Bessie Waldo (née Daniels)


In [9]:
passengers.tail()

,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname
1307,"Youssef, Mr. Gerios (Sam'aan)",45,"Hardîne, Syria",Cherbourg,"Wilkes-Barre, Pennsylvania, US",NaN,NaN,3,Youssef,Mr.,Gerios (Sam'aan)
1308,"Zajib Qiyamah, Miss Adal ""Jane""",15,"El Shweir, Syria",Cherbourg,"Brooklyn, New York, US",C,NaN,3,Zajib Qiyamah,Miss,"Adal ""Jane"""
1309,"Zakarian, Mr. Haroutyun Der",27,"Kiğı, Turkey",Cherbourg,"Brantford, Ontario, Canada",NaN,NaN,3,Zakarian,Mr.,Haroutyun Der
1310,"Zakarian, Mr. Mapri Der",22,"Kiğı, Turkey",Cherbourg,"Brantford, Ontario, Canada",NaN,304MB,3,Zakarian,Mr.,Mapri Der
1311,"Zimmermann, Mr. Leo",29,"Todtmoos, Germany",Southampton,"Saskatoon, Saskatchewan, Canada",NaN,NaN,3,Zimmermann,Mr.,Leo


In [10]:
print(passengers.title.value_counts(dropna=False))
female = ['Miss', 'Mrs.', 'Doña', 'Countess', 'Lady']
male = ['Mr.', 'Mr', 'Captain', 'Sir', 'Don', 'Major', 'The', 'Reverend', 'Father', 'Colonel', 'Major', 'Don', 'Master', 'Dr.']


Mr.         752
Miss        263
Mrs.        191
Master       58
NaN          17
Dr.           8
Colonel       5
Father        4
Reverend      3
The           2
Major         2
Don           1
Lady          1
Doña          1
Countess      1
Sir           1
Captain       1
Mr            1
Name: title, dtype: int64


In [11]:
passengers.loc[passengers.title == 'Major']

,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname
46,"Butt, Major Archibald Willingham",46,"Washington, D.C., US",Southampton,"Washington, D.C., US",NaN,NaN,1,Butt,Major,Archibald Willingham
229,"Peuchen, Major Arthur Godfrey",52,"Toronto, Ontario, Canada",Southampton,"Toronto, Ontario, Canada",6,NaN,1,Peuchen,Major,Arthur Godfrey


In [12]:
def set_sex(row):
#     print(row)
    if row.title in female:
        return 'female'
    elif row.title in male:
        return 'male'
    else:
        return 'unknown'
    
passengers['sex'] = passengers.apply(set_sex, axis=1)
print(passengers['sex'].value_counts())
passengers.head()

male       838
female     457
unknown     17
Name: sex, dtype: int64


,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname,sex
0,"Allen, Miss Elizabeth Walton",29,"St Louis, Missouri, US",Southampton,St Louis,NaN,NaN,1,Allen,Miss,Elizabeth Walton,female
1,"Allison, Mr. Hudson Creighton",30,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,135MB,1,Allison,Mr.,Hudson Creighton,male
2,"and chauffeur, Mr. George Swane[70]",19,NaN,Southampton,"Montreal, Quebec, Canada",NaN,294MB,1,and chauffeur,Mr.,George Swane[70],male
3,"and cook, Miss Amelia Mary ""Mildred"" Brown[70]",18,"London, England, UK",Southampton,"Montreal, Quebec, Canada",11,NaN,1,and cook,Miss,"Amelia Mary ""Mildred"" Brown[70]",female
4,"Allison, Mrs. Bessie Waldo (née Daniels)",25,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,NaN,1,Allison,Mrs.,Bessie Waldo (née Daniels),female


In [13]:
passengers.loc[339, 'sex'] = 'female'

In [14]:
passengers.loc[passengers.sex == 'unknown']

,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname,sex
428,NaN,24,"Detroit, Michigan, US",Southampton,"Detroit, Michigan, US",4,NaN,2,NaN,NaN,NaN,unknown
429,NaN,1,"Detroit, Michigan, US",Southampton,"Detroit, Michigan, US",4,NaN,2,NaN,NaN,NaN,unknown
911,NaN,27,NaN,Southampton,New York City,NaN,NaN,3,NaN,NaN,NaN,unknown
992,NaN,22,"Kauhajoki, Finland",Southampton,"Sudbury, Ontario, Canada",NaN,NaN,3,NaN,NaN,NaN,unknown
995,NaN,29,"Ikaalinen, Pirkanmaa, Finland",Southampton,"Glassport, Pennsylvania, US",NaN,NaN,3,NaN,NaN,NaN,unknown
1088,NaN,23,NaN,Southampton,"Peoria, Illinois, US",NaN,NaN,3,NaN,NaN,NaN,unknown
1089,NaN,22,"Mariestad, Västergötland, Sweden",Southampton,"Chicago, Illinois, US",C,NaN,3,NaN,NaN,NaN,unknown
1102,NaN,29,"Bjuv, Skåne, Sweden",Southampton,"Chicago, Illinois, US",NaN,206MB,3,NaN,NaN,NaN,unknown
1103,NaN,8,"Bjuv, Skåne, Sweden",Southampton,"Chicago, Illinois, US",NaN,NaN,3,NaN,NaN,NaN,unknown
1104,NaN,6,"Bjuv, Skåne, Sweden",Southampton,"Chicago, Illinois, US",NaN,NaN,3,NaN,NaN,NaN,unknown


In [15]:
print(passengers.shape)
passengers.dropna(axis=0, subset='name', inplace=True)
print(passengers.shape)

(1312, 12)
(1296, 12)


In [16]:
print(passengers['sex'].value_counts())
passengers.head()

male      838
female    458
Name: sex, dtype: int64


,name,age,hometown,boarded,destination,lifeboat,body,pclass,surname,title,fmname,sex
0,"Allen, Miss Elizabeth Walton",29,"St Louis, Missouri, US",Southampton,St Louis,NaN,NaN,1,Allen,Miss,Elizabeth Walton,female
1,"Allison, Mr. Hudson Creighton",30,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,135MB,1,Allison,Mr.,Hudson Creighton,male
2,"and chauffeur, Mr. George Swane[70]",19,NaN,Southampton,"Montreal, Quebec, Canada",NaN,294MB,1,and chauffeur,Mr.,George Swane[70],male
3,"and cook, Miss Amelia Mary ""Mildred"" Brown[70]",18,"London, England, UK",Southampton,"Montreal, Quebec, Canada",11,NaN,1,and cook,Miss,"Amelia Mary ""Mildred"" Brown[70]",female
4,"Allison, Mrs. Bessie Waldo (née Daniels)",25,"Montreal, Quebec, Canada",Southampton,"Montreal, Quebec, Canada",NaN,NaN,1,Allison,Mrs.,Bessie Waldo (née Daniels),female
